In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Create map
Map = geemap.Map(center=[8.5, -80], zoom=7)

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")

panama = countries.filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

roads = ee.FeatureCollection("projects/deforestation-495419/assets/Panama_OSM_Roads")

# Rasterize roads
roads_raster = ee.Image().byte().paint(roads, 1)

# Distance to nearest road in meters
distance_to_roads = (
    roads_raster
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(30)  # adjust if your dataset resolution differs
    .rename("dist_roads_m")
    .clip(panama)
)

vis = {
    "min": 0,
    "max": 5000,
    "palette": ["white", "blue", "green", "yellow", "red"]
}

Map.addLayer(panama, {}, "Panama boundary")

Map.addLayer(
    roads_raster,
    {"palette": ["black"]},
    "Roads (raster)"
)

Map.addLayer(
    distance_to_roads,
    vis,
    "Distance to Roads (m)"
)

Map 

Map(center=[8.5, -80], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…